# 🌬️ EPİAŞ Rüzgar Türbini Cut-off Analizi

Bu notebook ile:
1. EPİAŞ Şeffaflık'tan veri çekeceğiz
2. Rüzgar cut-off olaylarını tespit edeceğiz
3. Sonuçları analiz ve görselleştireceğiz


## 1. Kurulum ve Import


In [ ]:
# Gerekli kütüphaneleri yükle
# !pip install requests pandas numpy matplotlib

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Proje modülleri
from epias_client import EpiasClient
from wind_cutoff_analyzer import WindCutoffAnalyzer, visualize_cutoffs

# Görselleştirme ayarları
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("✅ Kütüphaneler yüklendi!")


## 2. EPİAŞ Credentials Ayarla

⚠️ **Önemli**: Şifrenizi notebook'a yazmayın, environment variable kullanın!


In [ ]:
# Credentials'ları environment variable olarak ayarla
# Yöntem 1: Direkt atama (test için)
# os.environ['EPIAS_USERNAME'] = 'email@domain.com'
# os.environ['EPIAS_PASSWORD'] = 'sifreniz'

# Yöntem 2: Interaktif input (daha güvenli)
if 'EPIAS_USERNAME' not in os.environ:
    os.environ['EPIAS_USERNAME'] = input("EPİAŞ Kullanıcı Adı: ")
    
if 'EPIAS_PASSWORD' not in os.environ:
    import getpass
    os.environ['EPIAS_PASSWORD'] = getpass.getpass("EPİAŞ Şifre: ")

print(f"✅ Kullanıcı: {os.environ['EPIAS_USERNAME'][:5]}...")


## 3. Veri Çekme

EPİAŞ'tan gerçek zamanlı üretim verisini çekiyoruz.


In [ ]:
# Client oluştur ve TGT al
client = EpiasClient()
tgt = client.get_tgt()
print(f"TGT: {tgt[:30]}...")


In [ ]:
# Son 5 ayın verisini çek (birkaç dakika sürebilir)
MONTHS_TO_FETCH = 5

files = client.fetch_multiple_months(months=MONTHS_TO_FETCH, export_type="CSV")

print(f"\n📁 İndirilen dosyalar:")
for f in files:
    print(f"   - {f.name}")


## 4. Veriyi Yükle ve Ön İşle


In [ ]:
# Analyzer oluştur ve tüm CSV'leri yükle
analyzer = WindCutoffAnalyzer()
df = analyzer.load_multiple_files("*.csv")

print(f"📊 Toplam satır: {len(df):,}")
print(f"\nSütunlar:")
for col in df.columns:
    print(f"   - {col}")


In [ ]:
# Veriyi ön işle
df = analyzer.preprocess_data()
print(f"🌬️ Rüzgar sütunu: {analyzer.wind_col}")
df.head()


## 5. Cut-off Tespiti


In [ ]:
# Cut-off parametreleri
analyzer.DROP_THRESHOLD_PERCENT = 50  # %50'den fazla düşüş = potansiyel cut-off
analyzer.MIN_PRODUCTION_MW = 10       # Min üretim (gürültüyü elemek için)
analyzer.RECOVERY_HOURS = 6           # Recovery penceresi

# Tespit et
cutoffs = analyzer.detect_cutoffs()
print(f"⚠️ Tespit edilen cut-off sayısı: {len(cutoffs)}")


In [ ]:
# En büyük 10 cut-off olayını göster
if len(cutoffs) > 0:
    display_cols = ['datetime', 'prev_production', 'production', 'drop_mw', 'drop_pct']
    available_cols = [c for c in display_cols if c in cutoffs.columns]
    print("📋 En büyük 10 cut-off olayı:")
    cutoffs.nlargest(10, 'drop_mw')[available_cols]


## 6. Analiz ve Rapor


In [ ]:
# Analiz ve rapor oluştur
analysis = analyzer.analyze_cutoff_patterns(cutoffs)
report = analyzer.generate_report(cutoffs, analysis, "cutoff_report.txt")
print(report)


## 7. Görselleştirme


In [ ]:
# Ana görselleştirme
if len(cutoffs) > 0 and analyzer.wind_col:
    visualize_cutoffs(df, cutoffs, analyzer.wind_col)


In [ ]:
# Cut-off olaylarını CSV'ye kaydet
if len(cutoffs) > 0:
    output_path = analyzer.export_cutoffs_to_csv(cutoffs)
    print(f"✅ Kaydedildi: {output_path}")


## 8. Özet Bulgular


In [ ]:
# Özet
if len(cutoffs) > 0 and 'datetime' in df.columns:
    print("📊 ÖZET BULGULAR")
    print("=" * 50)
    print(f"Analiz edilen dönem: {df['datetime'].min().date()} - {df['datetime'].max().date()}")
    print(f"Toplam saat: {len(df):,}")
    print(f"Tespit edilen cut-off: {len(cutoffs)}")
    print(f"Cut-off oranı: %{len(cutoffs)/len(df)*100:.2f}")
    print(f"Ortalama düşüş: {analysis.get('avg_drop_mw', 0):.1f} MW")
    print(f"Maksimum düşüş: {analysis.get('max_drop_mw', 0):.1f} MW")
else:
    print("ℹ️ Cut-off olayı tespit edilmedi veya veri yüklenemedi.")
